# 01 - Dataset, Calidad y EDA

**Objetivo:** Entender el dataset, validar calidad de datos, detectar problemas, y generar visualizaciones clave.

**Entregable:** Dataset entendido, validado, y listo para el pipeline de Integrante 2.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

from src.data.load_data import download_dataset, save_raw
from src.data.data_validation import (
    run_all_checks_raw,
    run_all_checks_clean,
    print_check_results,
)

print('Librerias cargadas.')

## 1. Carga del Dataset Crudo

In [ ]:
csv_path = download_dataset()
df_raw = pd.read_csv(csv_path)

print(f'Shape: {df_raw.shape}')
print(f'Columnas: {df_raw.columns.tolist()}')
df_raw.head(10)

In [ ]:
raw_path = save_raw(df_raw)
print(f'Crudo guardado en: {raw_path}')

## 2. Validacion de Calidad - Datos Crudos

In [ ]:
raw_checks = run_all_checks_raw(df_raw)
print_check_results('VALIDACION - DATOS CRUDOS', raw_checks)

## 3. Exploracion Inicial

In [ ]:
print('Tipos de datos:')
print(df_raw.dtypes)
print()
print(f'Filas: {df_raw.shape[0]}')
print(f'Columnas: {df_raw.shape[1]}')
print(f'Memoria: {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
print('Valores nulos por columna:')
nulls = df_raw.isnull().sum()
print(nulls[nulls > 0])
print(f'\nTotal filas con al menos un null: {df_raw.isnull().any(axis=1).sum()}')

In [ ]:
print('Filas duplicadas:', df_raw.duplicated().sum())
print('Empresas duplicadas:', df_raw['Company'].duplicated().sum())

## 4. Limpieza y Feature Engineering

In [ ]:
df = df_raw.copy()

# Limpiar nombre City\xa0
df.columns = [c.strip().replace('\xa0', '') for c in df.columns]

# valuation_b: limpiar $ y convertir a numero
df['valuation_b'] = (
    df['Valuation ($B)']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['valuation_b'] = pd.to_numeric(df['valuation_b'], errors='coerce')

# date_joined: convertir a datetime
df['date_joined'] = pd.to_datetime(df['Date Joined'], errors='coerce')
df['join_year'] = df['date_joined'].dt.year.astype('Int64')
df['join_month'] = df['date_joined'].dt.month.astype('Int64')

# investor_count: contar inversores
df['investor_count'] = df['Investors'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) and str(x).strip() else 0
)

# Renombrar columnas a formato limpio
df = df.rename(columns={
    'Company': 'company',
    'Country': 'country',
    'City': 'city',
    'Industry': 'industry',
    'Investors': 'investors',
})

print('Columnas finales:', df.columns.tolist())
df.head()

In [ ]:
print('Tipos de datos despues de limpieza:')
print(df[['valuation_b', 'date_joined', 'join_year', 'join_month', 'investor_count']].dtypes)
print()
print('Nulos despues de limpieza:')
print(df[['valuation_b', 'date_joined', 'join_year', 'join_month', 'investor_count']].isnull().sum())

## 5. Validacion - Datos Limpios

In [ ]:
clean_checks = run_all_checks_clean(df)
print_check_results('VALIDACION - DATOS LIMPIOS', clean_checks)

## 6. Analisis del Target (valuation_b)

In [ ]:
print('Estadisticas de valuation_b (miles de millones USD):')
print(df['valuation_b'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['valuation_b'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribucion de Valuation ($B)')
axes[0].set_xlabel('Valuation ($B)')
axes[0].set_ylabel('Frecuencia')
med = df['valuation_b'].median()
axes[0].axvline(med, color='red', linestyle='--', label='Mediana: {:.1f}'.format(med))
axes[0].legend()

axes[1].boxplot(df['valuation_b'].dropna(), vert=True)
axes[1].set_title('Boxplot de Valuation ($B)')
axes[1].set_ylabel('Valuation ($B)')

plt.tight_layout()
plt.show()

In [ ]:
print('Top 10 empresas mas valiosas:')
df.nlargest(10, 'valuation_b')[['company', 'valuation_b', 'country', 'industry']]

## 7. Distribucion por Pais

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

country_counts = df['country'].value_counts().head(10)
country_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Paises - Cantidad de Startups')
axes[0].set_xlabel('Cantidad')
axes[0].invert_yaxis()

country_val = df.groupby('country')['valuation_b'].mean().nlargest(10)
country_val.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 10 Paises - Valuacion Promedio ($B)')
axes[1].set_xlabel('Valuacion Promedio ($B)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 8. Distribucion por Industria

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ind_counts = df['industry'].value_counts().head(10)
ind_counts.plot(kind='barh', ax=axes[0], color='mediumseagreen')
axes[0].set_title('Top 10 Industrias - Cantidad de Startups')
axes[0].set_xlabel('Cantidad')
axes[0].invert_yaxis()

ind_val = df.groupby('industry')['valuation_b'].mean().nlargest(10)
ind_val.plot(kind='barh', ax=axes[1], color='goldenrod')
axes[1].set_title('Top 10 Industrias - Valuacion Promedio ($B)')
axes[1].set_xlabel('Valuacion Promedio ($B)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 9. Startups por Ano de Incorporacion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

year_counts = df['join_year'].value_counts().sort_index()
year_counts.plot(kind='bar', ax=axes[0], color='mediumpurple')
axes[0].set_title('Startups por Ano de Incorporacion')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Cantidad')

year_val = df.groupby('join_year')['valuation_b'].mean()
year_val.plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Valuacion Promedio por Ano')
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Valuacion Promedio ($B)')

plt.tight_layout()
plt.show()

## 10. Outliers en Valuation

In [ ]:
Q1 = df['valuation_b'].quantile(0.25)
Q3 = df['valuation_b'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

outliers = df[df['valuation_b'] > upper]
print('IQR: {:.2f}'.format(IQR))
print('Limite superior: {:.2f}'.format(upper))
print('Outliers detectados: {}'.format(len(outliers)))
print()
print('Empresas outliers (valuation > {:.2f}B):'.format(upper))
outliers[['company', 'valuation_b', 'country', 'industry']].head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(df['valuation_b'].dropna(), vert=True)
ax.axhline(upper, color='red', linestyle='--', label='Limite superior (IQR): {:.2f}'.format(upper))
ax.set_title('Boxplot con Outliers detectados')
ax.set_ylabel('Valuation ($B)')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Correlaciones

In [ ]:
num_cols = ['valuation_b', 'join_year', 'join_month', 'investor_count']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Matriz de Correlaciones')
plt.tight_layout()
plt.show()

## 12. Analisis de Nulos en Investors

In [ ]:
null_investors = df[df['investor_count'] == 0]
print('Filas con investors_count = 0 (originalmente NaN): {}'.format(len(null_investors)))
print()
if len(null_investors) > 0:
    print('Estas filas tienen Investors NaN en el original:')
    null_investors[['company', 'valuation_b', 'country', 'industry']].head(10)

## 13. Guardado del Dataset Limpio

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
processed_path = '../data/processed/dataset_clean.csv'
df.to_csv(processed_path, index=False)
print('Dataset limpio guardado en: {}'.format(processed_path))
print('Shape: {}'.format(df.shape))
print('Columnas: {}'.format(df.columns.tolist()))

## 14. Resumen de Hallazgos

### Calidad de Datos
- **1186 filas**, 7 columnas originales
- **18 nulos** en la columna Investors (1.5%)
- **0 duplicados** exactos
- **0 duplicados** por Company
- Columna `City` tiene caracteres `\xa0` en el nombre

### Target (valuation_b)
- Distribucion **muy sesgada a la derecha**
- Mediana: $1.6B, Promedio: $3.25B, Maximo: $140B (ByteDance)
- **~150 outliers** por encima del rango IQR

### Datos Faltantes
- 18 filas sin informacion de Investors (investor_count = 0)
- Sin nulos en otras columnas criticas

### Limitaciones del Dataset
1. Solo incluye startups hasta septiembre 2022
2. La valuacion es auto-declarada, no verificada
3. No hay informacion de fecha de fundacion (solo fecha de inclusion)
4. Los inversores estan como texto libre, no estructurado
5. La columna City tiene encoding inconsistente
6. No hay variables temporales para series de tiempo